> **Nota:** Todas as séries macroeconômicas utilizadas neste notebook possuem frequência **mensal**. Todos os cálculos, previsões e métricas são realizados considerando períodos mensais.

---

**Ficha Técnica do Modelo**

| Campo | Valor |
|-------|-------|
| **Modelo** | GARCH(p,q) — Generalized Autoregressive Conditional Heteroskedasticity |
| **Biblioteca** | `arch` 8.0.0 — `arch_model` |
| **Hiperparâmetros configurados** | `vol='GARCH'`, `p` ∈ {1..5}, `q` ∈ {1..5}, `mean` ∈ {`'Constant'`, `'Zero'`, `'AR'`} (lags=1 se AR), `dist` ∈ {`'normal'`, `'t'`, `'skewt'`} — 225 combinações testadas |
| **Busca de hiperparâmetros** | Sim — grid exaustivo (225 combinações), executado **uma vez** por série |
| **Critério de seleção** | AIC (in-sample, apenas dados de treino) |
| **Séries utilizadas** | Séries com total ≥ 84 observações (`len(s) < 60 + TEST_SIZE`) |
| **Horizonte** | 3 meses (`HORIZON = 3`) |
| **Protocolo de avaliação** | Walk-forward expansível, 24 meses de teste (`TEST_SIZE = 24`), janelas de 3 meses; previsões em log-retornos reconvertidas para nível |
| **Reprodutibilidade** | `np.random.seed(42)` (`SEED = 42`) |
| **Referência** | Bollerslev, T. (1986). Generalized Autoregressive Conditional Heteroskedasticity. *Journal of Econometrics*, 31(3), 307–327. |

---

# Forecasting com GARCH — Séries Macroeconômicas Brasileiras

Este notebook aplica modelos **GARCH** para previsão de volatilidade em séries macroeconômicas brasileiras.

- **Dados:** `base_economica_brasil.csv` (gerado pelo pipeline `Forecast_busca_bases_macroeconomia`)
- **Otimização:** Busca automática da melhor ordem GARCH(p,q) com p=1..5, q=1..5, modelo de média e distribuição via AIC
- **Avaliação:** MAE, RMSE e MAPE sobre previsão de volatilidade vs. volatilidade histórica
- **Saídas:** `resultados_garch.csv` e `previsoes_garch.csv` no formato padrão para comparação entre modelos

### ⚠️ Nota Metodológica: Uso do GARCH para Previsão de Nível

O modelo GARCH (Generalized ARCH) foi proposto por Bollerslev (1986) como uma generalização do ARCH de Engle (1982), adicionando termos autorregressivos à equação da **variância condicional**. Seu uso típico é para modelagem e previsão de volatilidade.

Neste trabalho, a previsão pontual de **nível** (valor da série) é extraída da **equação da média** do modelo estimado. A especificação da média é selecionada por AIC dentre três opções: **Constant** (média constante), **Zero** (média zero) e **AR(1)** (autorregressivo de ordem 1). O modelo opera sobre log-retornos, e as previsões da equação da média (`forecast.mean`) são convertidas de volta para níveis via:

$$\hat{Y}_{t+h} = Y_t \cdot \exp\!\left(\sum_{i=1}^{h} \hat{r}_i\right)$$

Essa é uma abordagem **atípica** — o uso convencional do GARCH é para previsão de volatilidade, não de média. No entanto, a inclusão se justifica como **benchmark** na comparação com métodos tradicionais de forecasting de média, permitindo avaliar se a modelagem conjunta de média e variância condicional contribui para a previsão de nível.

## 1. Instalação de Dependências

In [2]:
!pip install arch

# Importação das bibliotecas
import pandas as pd
import numpy as np

# ── Semente global para reprodutibilidade ──
SEED = 42
np.random.seed(SEED)
import matplotlib.pyplot as plt
from arch import arch_model

# Configurações de exibição



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Carregamento dos Dados

In [2]:
# Carregar todas as séries macroeconômicas do pipeline consolidado
# Todas as séries são mensais.
df = pd.read_csv('base_economica_brasil.csv', index_col=0, parse_dates=True)
ALL_SERIES = [col for col in df.columns if col not in ['PIM', 'IPCA_mensal']]
print(f'Séries macroeconômicas disponíveis ({len(ALL_SERIES)}): {ALL_SERIES}')

Séries macroeconômicas disponíveis (35): ['IBC_Br', 'Selic', 'Cambio_USDBRL', 'Desemprego', 'Brent_USD', 'Soja_USD', 'Minerio_USD', 'Ibovespa', 'ICC_FGV', 'Credito_Total', 'Inadimplencia', 'Massa_Salarial', 'CPI_USA', 'Prod_Ind_USA', 'Cafe_USD', 'Ouro_USD', 'GasNatural_USD', 'Cobre_USD', 'ETF_Emergentes', 'IGP_DI', 'INCC', 'ICE_Empresarial', 'Housing_Starts_EUA', 'Dollar_Index_Fed', 'PMS_Volume', 'PMC_Ampliado', 'IGPM', 'INPC', 'M2', 'Divida_PIB', 'Vendas_Varejo', 'Balanca_Comercial', 'NUCI_FGV', 'EAI_Emprego_Ind', 'SP500']


## 3. Configuração do Experimento

In [3]:
from arch import arch_model
from sklearn.metrics import mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

HORIZON = 3
TEST_SIZE = 24

def classificar_mape(mape):
    if mape < 10:
        return '⭐ Excelente'
    elif mape < 20:
        return '✅ Muito Bom'
    elif mape < 30:
        return '👍 Bom'
    elif mape < 50:
        return '⚠️ Regular'
    else:
        return '❌ Difícil'

def select_best_garch_by_aic(returns):
    """
    Seleciona melhor GARCH(p,q) por AIC nos dados de treino.
    Testa p=1..5, q=1..5, 3 médias, 3 distribuições = 225 combinações.
    """
    best_aic = np.inf
    best_params = None
    total_tested = 0
    total_ok = 0

    mean_models = ['Constant', 'Zero', 'AR']
    distributions = ['normal', 't', 'skewt']

    for p in range(1, 6):
        for q in range(1, 6):
            for mean in mean_models:
                for dist in distributions:
                    total_tested += 1
                    try:
                        kw = {'mean': mean, 'vol': 'GARCH', 'p': p, 'q': q,
                              'dist': dist}
                        if mean == 'AR':
                            kw['lags'] = 1
                        model = arch_model(returns, **kw)
                        fit = model.fit(disp='off')
                        if fit.aic < best_aic:
                            best_aic = fit.aic
                            best_params = {'p': p, 'q': q, 'mean': mean, 'dist': dist}
                        total_ok += 1
                    except Exception:
                        continue

    return best_params, total_tested, total_ok

def calc_metrics(y_true, y_pred):
    """Calcula MAE, RMSE e MAPE (mesmo padrão do XGBoost)."""
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-8))) * 100
    return mae, rmse, mape

# ── Loop principal ──
resultados = []
previsoes_rows = []

print(f'{"="*70}')
print(f'GARCH — Walk-Forward Level Forecasting (TEST={TEST_SIZE}, H={HORIZON})')
print(f'{"="*70}\n')

for serie in ALL_SERIES:
    print(f'▶ {serie}')
    s = df[serie].dropna()

    if len(s) < 60 + TEST_SIZE:
        print(f'  ⏭ Poucos dados ({len(s)} obs). Pulando.\n')
        continue

    # Seleção de modelo por AIC (apenas dados de treino)
    train_returns = np.log(s.iloc[:-TEST_SIZE]).diff().dropna()
    best_params, n_tested, n_ok = select_best_garch_by_aic(train_returns)
    if best_params is None:
        print(f'  ❌ Nenhum modelo convergiu ({n_tested} tentativas).\n')
        continue

    dist_label = {'normal': 'Normal', 't': 'Student-t', 'skewt': 'Skew-t'}[best_params['dist']]
    mean_label = {'Constant': 'C', 'Zero': '0', 'AR': 'AR(1)'}[best_params['mean']]
    model_label = f"GARCH({best_params['p']},{best_params['q']})|{mean_label}|{dist_label}"

    # Walk-forward: prever NÍVEIS (não volatilidade)
    all_preds = []
    all_actuals = []
    all_dates = []

    for start in range(0, TEST_SIZE, HORIZON):
        split_idx = len(s) - TEST_SIZE + start
        n_steps = min(HORIZON, TEST_SIZE - start)

        # Training expandido (com dados de teste já revelados)
        cur_train = s.iloc[:split_idx]
        cur_returns = np.log(cur_train).diff().dropna()
        last_level = cur_train.iloc[-1]

        # Valores reais para comparação
        actual_levels = s.iloc[split_idx:split_idx + n_steps]

        try:
            kw = {'mean': best_params['mean'], 'vol': 'GARCH',
                  'p': best_params['p'], 'q': best_params['q'],
                  'dist': best_params['dist']}
            if best_params['mean'] == 'AR':
                kw['lags'] = 1
            model = arch_model(cur_returns, **kw)
            fit = model.fit(disp='off')
            fc = fit.forecast(horizon=n_steps)
            mean_ret = fc.mean.values[-1, :n_steps]

            # Converter retornos previstos → níveis
            cum_ret = np.cumsum(mean_ret)
            pred_levels = last_level * np.exp(cum_ret)

            all_preds.extend(pred_levels)
            all_actuals.extend(actual_levels.values)
            all_dates.extend(actual_levels.index)
        except Exception:
            continue

    if len(all_preds) == 0:
        print(f'  ❌ Walk-forward falhou para {serie}.\n')
        continue

    preds_arr = np.array(all_preds)
    actuals_arr = np.array(all_actuals)
    mae, rmse, mape = calc_metrics(actuals_arr, preds_arr)

    classif = classificar_mape(mape)
    resultados.append({
        'Serie': serie,
        'MAE': round(mae, 6),
        'RMSE': round(rmse, 6),
        'MAPE': round(mape, 2),
        'N_Pontos': len(all_preds),
        'Classificacao': classif,
        'Modelo': model_label
    })

    # Previsões para CSV
    for i in range(len(all_preds)):
        previsoes_rows.append({
            'Serie': serie,
            'Data': str(all_dates[i])[:10],
            'Previsao': all_preds[i]
        })

    print(f'  Testados: {n_tested} | Convergidos: {n_ok} | Melhor: {model_label}')
    print(f'  Pontos previstos: {len(all_preds)}/{TEST_SIZE}')
    print(f'  MAPE: {mape:.2f}% | {classif}\n')

# Salvar CSVs no formato padrão
results_df = pd.DataFrame(resultados)
results_df[['Serie','MAE','RMSE','MAPE','N_Pontos','Classificacao']].to_csv('resultados_garch.csv', index=False)

prev_df = pd.DataFrame(previsoes_rows)
prev_df[['Serie','Data','Previsao']].to_csv('previsoes_garch.csv', index=False)

print(f'{"="*70}')
print(f'✅ Resultados salvos: resultados_garch.csv ({len(resultados)} séries)')
print(f'✅ Previsões salvas: previsoes_garch.csv ({len(previsoes_rows)} registros)')
print(f'{"="*70}')

GARCH — Walk-Forward Level Forecasting (TEST=12, H=3)

▶ IBC_Br


C:\Users\phill\AppData\Local\Temp\ipykernel_22000\1176175619.py:45: ConvergenceWarning: The optimizer returned code 8. The message is:
Positive directional derivative for linesearch
See scipy.optimize.fmin_slsqp for code meaning.

  fit = model.fit(disp='off')
C:\Users\phill\AppData\Local\Temp\ipykernel_22000\1176175619.py:45: ConvergenceWarning: The optimizer returned code 8. The message is:
Positive directional derivative for linesearch
See scipy.optimize.fmin_slsqp for code meaning.

  fit = model.fit(disp='off')
C:\Users\phill\AppData\Local\Temp\ipykernel_22000\1176175619.py:45: ConvergenceWarning: The optimizer returned code 8. The message is:
Positive directional derivative for linesearch
See scipy.optimize.fmin_slsqp for code meaning.

  fit = model.fit(disp='off')
C:\Users\phill\AppData\Local\Temp\ipykernel_22000\1176175619.py:45: ConvergenceWarning: The optimizer returned code 8. The message is:
Positive directional derivative for linesearch
See scipy.optimize.fmin_slsqp for c

  Testados: 225 | Convergidos: 225 | Melhor: GARCH(2,4)|AR(1)|Skew-t
  Pontos previstos: 12/12
  MAPE: 2.94% | ⭐ Excelente

▶ Selic


C:\Users\phill\AppData\Local\Temp\ipykernel_22000\1176175619.py:45: ConvergenceWarning: The optimizer returned code 4. The message is:
Inequality constraints incompatible
See scipy.optimize.fmin_slsqp for code meaning.

  fit = model.fit(disp='off')
C:\Users\phill\AppData\Local\Temp\ipykernel_22000\1176175619.py:45: ConvergenceWarning: The optimizer returned code 8. The message is:
Positive directional derivative for linesearch
See scipy.optimize.fmin_slsqp for code meaning.

  fit = model.fit(disp='off')
C:\Users\phill\AppData\Local\Temp\ipykernel_22000\1176175619.py:45: ConvergenceWarning: The optimizer returned code 8. The message is:
Positive directional derivative for linesearch
See scipy.optimize.fmin_slsqp for code meaning.

  fit = model.fit(disp='off')
C:\Users\phill\AppData\Local\Temp\ipykernel_22000\1176175619.py:45: ConvergenceWarning: The optimizer returned code 4. The message is:
Inequality constraints incompatible
See scipy.optimize.fmin_slsqp for code meaning.

  fit = 

  Testados: 225 | Convergidos: 225 | Melhor: GARCH(3,5)|0|Student-t
  Pontos previstos: 12/12
  MAPE: 4.13% | ⭐ Excelente

▶ Cambio_USDBRL
  Testados: 225 | Convergidos: 225 | Melhor: GARCH(1,1)|C|Normal
  Pontos previstos: 12/12
  MAPE: 3.85% | ⭐ Excelente

▶ Desemprego


...

  Testados: 225 | Convergidos: 225 | Melhor: GARCH(2,1)|C|Normal
  Pontos previstos: 12/12
  MAPE: 0.85% | ⭐ Excelente

▶ Vendas_Varejo
  Testados: 225 | Convergidos: 225 | Melhor: GARCH(1,1)|AR(1)|Student-t
  Pontos previstos: 12/12
  MAPE: 9.68% | ⭐ Excelente

▶ Balanca_Comercial
  Testados: 225 | Convergidos: 225 | Melhor: GARCH(1,1)|AR(1)|Student-t
  Pontos previstos: 12/12
  MAPE: 83.14% | ❌ Difícil

▶ NUCI_FGV


C:\Users\phill\AppData\Local\Temp\ipykernel_22000\1176175619.py:45: ConvergenceWarning: The optimizer returned code 8. The message is:
Positive directional derivative for linesearch
See scipy.optimize.fmin_slsqp for code meaning.

  fit = model.fit(disp='off')
C:\Users\phill\AppData\Local\Temp\ipykernel_22000\1176175619.py:45: ConvergenceWarning: The optimizer returned code 8. The message is:
Positive directional derivative for linesearch
See scipy.optimize.fmin_slsqp for code meaning.

  fit = model.fit(disp='off')
C:\Users\phill\AppData\Local\Temp\ipykernel_22000\1176175619.py:45: ConvergenceWarning: The optimizer returned code 8. The message is:
Positive directional derivative for linesearch
See scipy.optimize.fmin_slsqp for code meaning.

  fit = model.fit(disp='off')
C:\Users\phill\AppData\Local\Temp\ipykernel_22000\1176175619.py:45: ConvergenceWarning: The optimizer returned code 8. The message is:
Positive directional derivative for linesearch
See scipy.optimize.fmin_slsqp for c

  Testados: 225 | Convergidos: 225 | Melhor: GARCH(1,2)|C|Skew-t
  Pontos previstos: 12/12
  MAPE: 2.56% | ⭐ Excelente

▶ EAI_Emprego_Ind


C:\Users\phill\AppData\Local\Temp\ipykernel_22000\1176175619.py:45: ConvergenceWarning: The optimizer returned code 9. The message is:
Iteration limit reached
See scipy.optimize.fmin_slsqp for code meaning.

  fit = model.fit(disp='off')
C:\Users\phill\AppData\Local\Temp\ipykernel_22000\1176175619.py:45: ConvergenceWarning: The optimizer returned code 9. The message is:
Iteration limit reached
See scipy.optimize.fmin_slsqp for code meaning.

  fit = model.fit(disp='off')
C:\Users\phill\AppData\Local\Temp\ipykernel_22000\1176175619.py:45: ConvergenceWarning: The optimizer returned code 9. The message is:
Iteration limit reached
See scipy.optimize.fmin_slsqp for code meaning.

  fit = model.fit(disp='off')
C:\Users\phill\AppData\Local\Temp\ipykernel_22000\1176175619.py:45: ConvergenceWarning: The optimizer returned code 9. The message is:
Iteration limit reached
See scipy.optimize.fmin_slsqp for code meaning.

  fit = model.fit(disp='off')
C:\Users\phill\AppData\Local\Temp\ipykernel_2200

  Testados: 225 | Convergidos: 225 | Melhor: GARCH(1,1)|AR(1)|Student-t
  Pontos previstos: 12/12
  MAPE: 196.20% | ❌ Difícil

▶ SP500
  Testados: 225 | Convergidos: 225 | Melhor: GARCH(1,1)|AR(1)|Skew-t
  Pontos previstos: 12/12
  MAPE: 3.31% | ⭐ Excelente

✅ Resultados salvos: resultados_garch.csv (34 séries)
✅ Previsões salvas: previsoes_garch.csv (408 registros)


## 4. Resultados e Métricas

In [4]:
# Ranking das séries por MAPE (do melhor para o pior)
ranking = results_df.sort_values('MAPE')
print('=' * 70)
print('RANKING GARCH \u2014 Previs\u00e3o de Volatilidade')
print('=' * 70)
for i, row in ranking.iterrows():
    print(f"  {row['Classificacao']:20s} | {row['Serie']:30s} | MAPE: {row['MAPE']:7.2f}% | {row['Modelo']}")
print('=' * 70)
print(f"\nMAPE m\u00e9dio geral: {ranking['MAPE'].mean():.2f}%")

RANKING GARCH — Previsão de Volatilidade
  ⭐ Excelente          | CPI_USA                        | MAPE:    0.12% | GARCH(1,1)|AR(1)|Normal
  ⭐ Excelente          | M2                             | MAPE:    0.55% | GARCH(1,1)|C|Normal
  ⭐ Excelente          | Divida_PIB                     | MAPE:    0.85% | GARCH(2,1)|C|Normal
  ⭐ Excelente          | Credito_Total                  | MAPE:    1.01% | GARCH(1,1)|C|Normal
  ⭐ Excelente          | Massa_Salarial                 | MAPE:    1.19% | GARCH(1,1)|AR(1)|Normal
  ⭐ Excelente          | Prod_Ind_USA                   | MAPE:    1.40% | GARCH(3,1)|C|Student-t
  ⭐ Excelente          | Dollar_Index_Fed               | MAPE:    1.85% | GARCH(1,1)|0|Normal
  ⭐ Excelente          | NUCI_FGV                       | MAPE:    2.56% | GARCH(1,2)|C|Skew-t
  ⭐ Excelente          | IBC_Br                         | MAPE:    2.94% | GARCH(2,4)|AR(1)|Skew-t
  ⭐ Excelente          | SP500                          | MAPE:    3.31% | GARCH(1,1)|AR(

## 5. Visualização: Ranking MAPE por Série

In [ ]:
# ── Gráfico: Ranking MAPE por Série ──
sorted_df = results_df.sort_values('MAPE')

fig, ax = plt.subplots(figsize=(12, 8))

cores = ['#2ecc71' if m < 10 else '#3498db' if m < 20 else '#f39c12' if m < 30 else '#e74c3c'
         for m in sorted_df['MAPE']]

bars = ax.barh(range(len(sorted_df)), sorted_df['MAPE'],
               color=cores, edgecolor='white', height=0.7)
ax.set_yticks(range(len(sorted_df)))
ax.set_yticklabels(sorted_df['Serie'])
ax.invert_yaxis()
ax.set_xlabel('MAPE (%)')
ax.set_title(f'GARCH — MAPE por Série\n(Walk-Forward, h={HORIZON}, teste={TEST_SIZE} meses)',
             fontsize=13, fontweight='bold')
ax.axvline(x=sorted_df['MAPE'].mean(), color='red', linestyle='--',
           label=f'Média: {sorted_df["MAPE"].mean():.1f}%')
ax.legend(loc='lower right')

for i, (bar, val) in enumerate(zip(bars, sorted_df['MAPE'])):
    ax.text(val + 0.5, i, f'{val:.1f}%', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('garch_mape_por_serie.png', dpi=150, bbox_inches='tight')
plt.show()
print("📊 Gráfico salvo: garch_mape_por_serie.png")

## 6. Exportação de Resultados